In [ ]:
pip install pandas openpyxl # INSTALAÇÃO DO PANDAS PARA COMEÇO DAS ATIVIDADES

In [44]:
# REALIZAÇÃO DA SUBIDA DOS ARQUIVOS
import pandas as pd
# Removed import io and from google.colab import files as they are no longer needed for direct file reading

# 1. Removed files.upload() call as files are already in /content/

# 2. Carrega os arquivos para os DataFrames
df_clientes_xlsx = pd.read_excel('/content/clientes.xlsx') # Changed to read_excel
df_clientes_csv = pd.read_csv('/content/clientes.csv', sep=';', encoding='latin1')
df_auto = pd.read_csv('/content/auto.csv', sep=';')

print("--- Arquivos carregados com sucesso! ---")

--- Arquivos carregados com sucesso! ---


In [ ]:
# EXIBIÇÃO DOS PRIMEIROS DADOS DE CADA PLANILHA
# 3. Exibir o formato inicial das tabelas
print("--- Clientes (XLSX) ---")
display(df_clientes_xlsx.head(2))

print("--- Clientes (CSV) ---")
display(df_clientes_csv.head(2))

print("--- Automóveis (CSV) ---")
display(df_auto.head(2))

--- Clientes (XLSX) ---


,id_client;age;monthly_income;city;state;gender;education_level
0,1001;50;14544;Muriaé;MG;NA;NA
1,1002;31;1230;Araxá;MG;male;NA


--- Clientes (CSV) ---


,id_client,age,monthly_income,city,state,gender,education_level
0,1001,50,14544,Muriaé,MG,NaN,NaN
1,1002,31,1230,Araxá,MG,male,NaN


--- Automóveis (CSV) ---


,id_client,auto_brand,auto_model,auto_year,auto_value,id_cliente
0,1993Au16811,Audi,100 2.8 V6,1993,40000,28596
1,1994Au13275,Audi,100 2.8 V6 Avant,1994,13873,1048


In [17]:
# RESOLUÇÃO DA QUESTÃO 2
# 1. Unificar as duas bases de clientes e remover duplicatas
df_clientes = pd.concat([df_clientes_xlsx, df_clientes_csv]).drop_duplicates()

# 2. Limpar espaços vazios e padronizar textos de estado/cidade
df_clientes['state'] = df_clientes['state'].astype(str).str.strip().str.upper()

# 3. Tratar a coluna de salário (caso esteja salva como texto/moeda)
# The column name is 'monthly_income', not 'salario'
if 'monthly_income' in df_clientes.columns and df_clientes['monthly_income'].dtype == 'object':
    df_clientes['monthly_income'] = (
        df_clientes['monthly_income']
        .astype(str)
        .str.replace('R$', '', regex=False)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
        .str.strip()
        .astype(float)
    )

# 4. Definir os estados da Região Sudeste
estados_sudeste = ['SP', 'RJ', 'MG', 'ES']

# 5. Filtrar os clientes do Sudeste e calcular a média salarial por estado
# Use 'monthly_income' for salary calculations
media_sudeste = (
    df_clientes[df_clientes['state'].isin(estados_sudeste)]
    .groupby('state')['monthly_income'] # Changed to monthly_income
    .mean()
    .round(2)
    .reset_index()
)

# Renomear as colunas para exibição limpa
media_sudeste.columns = ['Estado', 'Média Salarial']

print("=== Média Salarial dos Clientes por Estado (Sudeste) ===")
display(media_sudeste)

=== Média Salarial dos Clientes por Estado (Sudeste) ===


,Estado,Média Salarial
0,ES,11867.91
1,MG,11498.02
2,RJ,11462.05
3,SP,11577.04


In [22]:
# RESOLUÇÃO DO EXERCICIO 3
import pandas as pd

# 1. Unificar as bases
df_clientes = pd.concat([df_clientes_xlsx, df_clientes_csv]).drop_duplicates()

# Padronizar nomes das colunas
df_clientes.columns = df_clientes.columns.str.strip().str.lower()

# 2. Limpar a coluna 'estado' (remover nulos, espaços e converter para maiúsculo)
df_clientes['state'] = df_clientes['state'].astype(str).str.strip().str.upper()

# 3. FILTRO DE LIMPEZA: Remover valores nulos, 'NAN', 'NONE' ou siglas que não tenham 2 letras
df_clientes_limpo = df_clientes[
    df_clientes['state'].str.len() == 2
].copy()

# 4. Tratar a coluna 'salario'
if df_clientes_limpo['monthly_income'].dtype == 'object':
    df_clientes_limpo['monthly_income'] = (
        df_clientes_limpo['monthly_income']
        .astype(str)
        .str.replace('R$', '', regex=False)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
        .str.strip()
        .astype(float)
    )

# 5. Agrupar apenas pelos estados válidos da base
extremos_salario = (
    df_clientes_limpo
    .groupby('state')['monthly_income']
    .agg(Maior='max', Menor='min')
    .round(2)
    .reset_index()
)

extremos_salario.columns = ['Estado', 'Maior', 'Menor']

print(f"Total de estados válidos encontrados: {len(extremos_salario)}")
print("\n=== Tabela Oficial da Questão 3 (Sem linhas nulas) ===")
display(extremos_salario)

Total de estados válidos encontrados: 27

=== Tabela Oficial da Questão 3 (Sem linhas nulas) ===


,Estado,Maior,Menor
0,AC,22001.0,1042.0
1,AL,22056.0,1213.0
2,AM,22032.0,1032.0
3,AP,21870.0,1117.0
4,BA,22139.0,1037.0
5,CE,22011.0,1047.0
6,DF,101200.0,1039.0
7,ES,21938.0,1042.0
8,GO,22135.0,700.0
9,MA,22100.0,1038.0


In [25]:
# RESOLUÇÃO DA 4
import pandas as pd

# 1. Verificar se existe coluna de 'idade' ou 'data_nascimento'
colunas = df_clientes_limpo.columns.tolist()
print("Colunas disponíveis na base:", colunas)

# 2. Lógica para calcular a idade (se for data de nascimento) ou usar a coluna existente
if 'age' in colunas:
    df_clientes_limpo['idade_calc'] = pd.to_numeric(df_clientes_limpo['age'], errors='coerce')

elif 'data_nascimento' in colunas or 'dt_nascimento' in colunas:
    # Identifica o nome da coluna de data
    col_data = 'data_nascimento' if 'data_nascimento' in colunas else 'dt_nascimento'

    # Converte para formato de data
    df_clientes_limpo[col_data] = pd.to_datetime(df_clientes_limpo[col_data], dayfirst=True, errors='coerce')

    # Calcula a idade em anos considerando a data de hoje (ou ano de referência do teste)
    hoje = pd.to_datetime('today')
    df_clientes_limpo['idade_calc'] = (hoje - df_clientes_limpo[col_data]).dt.days // 365.25

# 3. Cálculo do percentual de clientes acima de 60 anos
total_clientes = len(df_clientes_limpo)
clientes_acima_60 = (df_clientes_limpo['idade_calc'] > 60).sum()

percentual = (clientes_acima_60 / total_clientes) * 100

print("\n================ RESULTADO DA QUESTÃO 4 ================")
print(f"Total de clientes analisados: {total_clientes}")
print(f"Clientes com mais de 60 anos: {clientes_acima_60}")
print(f"Percentual: {percentual:.2f}%")
print("========================================================")

Colunas disponíveis na base: ['id_client;age;monthly_income;city;state;gender;education_level', 'id_client', 'age', 'monthly_income', 'city', 'state', 'gender', 'education_level']

================ RESULTADO DA QUESTÃO 4 ================
Total de clientes analisados: 35019
Clientes com mais de 60 anos: 1596
Percentual: 4.56%


In [40]:
# CONTAGEM DOS CARROS
# Ver quais marcas existem na tabela de automóveis e a contagem de cada uma
print("Marcas encontradas na base de carros:")
print(df_auto['auto_brand'].value_counts(dropna=False).head(10))

Marcas encontradas na base de carros:
auto_brand
Fiat               7645
GM - Chevrolet     7223
VW - VolksWagen    6599
Ford               3759
Renault            2105
Honda              1848
Peugeot            1225
Citroën             932
Hyundai             925
Toyota              677
Name: count, dtype: int64


In [41]:
# ANEXO SOBRE IDS DAS COLUNAS DO ARQUIVO AUTO.CSV
print("--- Amostra ID Clientes ---")
print(df_clientes_limpo['id_client'].dropna().head(5).tolist())

print("\n--- Amostra ID Carros ---")
print(df_auto['id_client'].dropna().head(5).tolist())

--- Amostra ID Clientes ---
['1001.0', '1002.0', '1003.0', '1004.0', '1005.0']

--- Amostra ID Carros ---
['1993Au16811', '1994Au13275', '1998Pe10697', '1999Pe13343', '2000Pe13776']


In [42]:
# RESOLUÇÃO DA 5
import pandas as pd

# 1. Padronizar a base de automóveis
df_auto.columns = df_auto.columns.str.strip().str.lower()

# Ajusta a chave se necessário
if 'auto_id' in df_auto.columns:
    df_auto.rename(columns={'auto_id': 'id_client'}, inplace=True)

# 2. Higienização avançada das chaves de ligação (ID)
# Converte para string, extrai apenas os números e remove zeros à esquerda/decimais
df_clientes_limpo['id_join'] = (
    df_clientes_limpo['id_client']
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.extract(r'(\d+)', expand=False) # Mantém apenas os dígitos numéricos
)

df_auto['id_join'] = (
    df_auto['id_client']
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.extract(r'(\d+)', expand=False)
)

# 3. Fazer o cruzamento pelas chaves limpas
df_completo = pd.merge(df_clientes_limpo, df_auto, on='id_join', how='inner')

# Identificar colunas corretas de Cidade e Marca
col_cidade = 'city' if 'city' in df_completo.columns else 'cidade'
col_marca = 'auto_brand' if 'auto_brand' in df_completo.columns else 'marca'

# 4. Normalizar texto da marca e filtrar FORD
df_completo['marca_clean'] = df_completo[col_marca].astype(str).str.strip().str.upper()
df_ford = df_completo[df_completo['marca_clean'].str.contains('FORD', na=False)]

# 5. Agrupar por cidade e obter a maior
ranking_ford = (
    df_ford.groupby(col_cidade)['marca_clean']
    .count()
    .reset_index(name='Quantidade')
    .sort_values(by='Quantidade', ascending=False)
)

print("================ RESULTADO DA QUESTÃO 5 ================")
if not ranking_ford.empty:
    cidade_top = ranking_ford.iloc[0][col_cidade]
    qtd_top = ranking_ford.iloc[0]['Quantidade']
    print(f"Cidade com mais carros Ford: {cidade_top}")
    print(f"Quantidade: {qtd_top}")
    print("========================================================")
    print("\n--- Top 5 Cidades com mais carros Ford ---")
    display(ranking_ford.head(5))
else:
    print("O merge ainda falhou. Verifique os prints do Passo 1 para ajustarmos o ID.")

================ RESULTADO DA QUESTÃO 5 ================
Cidade com mais carros Ford: Brasília
Quantidade: 428

--- Top 5 Cidades com mais carros Ford ---


,city,Quantidade
1,Brasília,428
2,Camaçari,336
9,Jaboticabal,296
23,Sumaré,294
20,Santa Maria,288


In [53]:
# Instalação da biblioteca unidecode
# RESOLUÇÃO DA 6
!pip install unidecode

import pandas as pd
import re
from unidecode import unidecode
from google.colab import files

# 1. Obter a série de cidades do RS da sua base
df_rs = df_clientes_limpo[df_clientes_limpo['state'] == 'RS'].copy() # Corrigido 'estado' para 'state'
col_cidade = 'cidade' if 'cidade' in df_rs.columns else 'city'

# 2. FUNÇÃO DE HIGIENIZAÇÃO DE NOMES DE CIDADES
def limpar_nome_cidade(nome):
    if pd.isna(nome) or str(nome).strip().lower() in ['nan', 'none', '', 'null']:
        return None

    texto = str(nome).strip()

    # Remover múltiplos espaços internos
    texto = re.sub(r'\s+', ' ', texto)

    # Remover o sufixo " RS" se existir no final do nome
    texto = re.sub(r'\s+RS$', '', texto, flags=re.IGNORECASE)

    # Dicionário de padronizações e correções de apelidos/erros
    correcoes_mapeadas = {
        'POA': 'Porto Alegre',
        'PORTO ALEGRE': 'Porto Alegre',
        'PORTO  ALEGRE': 'Porto Alegre',
        'ITORRESGREJINHA': 'Igrejinha',
        'VIAMAO': 'Viamão',
        'GRAVATAI': 'Gravataí',
        'BENTO GONCALVES': 'Bento Gonçalves',
        'BENTO GONCALVE': 'Bento Gonçalves',
        'ESTANCIA VELHA': 'Estância Velha',
        'CACHOEIRA DO SUL': 'Cachoeira do Sul',
        'CAMPO BOM': 'Campo Bom',
        'CANELA': 'Canela',
        'CANGUÇU': 'Canguçu',
        'CANGUCU': 'Canguçu',
        'CANOAS': 'Canoas',
        'CARAZINHO': 'Carazinho',
        'CAXIAS DO SUL': 'Caxias do Sul',
        'CONSTANTINA': 'Constantina',
        'ERECHIM': 'Erechim',
        'GIRUA': 'Giruá',
        'GRAMADO': 'Gramado',
        'MARAU': 'Marau',
        'NOVO HAMBURGO': 'Novo Hamburgo',
        'PASSO FUNDO': 'Passo Fundo',
        'PELOTAS': 'Pelotas',
        'RIO GRANDE': 'Rio Grande',
        'SANTA CRUZ DO SUL': 'Santa Cruz do Sul',
        'SANTO ANTONIO DA PATRULHA': 'Santo Antônio da Patrulha',
        'SAO SEPE': 'São Sepé',
        'SOBRADINHO': 'Sobradinho',
        'TRES DE MAIO': 'Três de Maio',
        'VACARIA': 'Vacaria',
        'VIADUTOS': 'Viadutos',
        'BOA VISTA DO BURICA': 'Boa Vista do Buricá'
    }

    # Aplicar a correção direta se bater no dicionário
    texto_upper = texto.upper()
    if texto_upper in correcoes_mapeadas:
        return correcoes_mapeadas[texto_upper]

    # Title Case padrão (Ex: "porto alegre" -> "Porto Alegre")
    texto_title = texto.title()

    # Ajustar preposições em minúsculo (do, da, dos, das, de)
    preposicoes = [' De ', ' Do ', ' Da ', ' Dos ', ' Das ', ' E ']
    for prep in preposicoes:
        texto_title = texto_title.replace(prep, prep.lower())

    return texto_title

# 3. Aplicar a limpeza na coluna de cidades
df_rs['cidade_limpa'] = df_rs[col_cidade].apply(limpar_nome_cidade)

# 4. Agrupar por versão "unidecode" (sem acentos/caixa alta) para consolidar duplicatas com grafias levemente diferentes
# Primeiro, filtra as linhas onde 'cidade_limpa' não é nulo para evitar o erro 'NoneType' no unidecode
df_rs_cleaned = df_rs.dropna(subset=['cidade_limpa'])

df_cidades_unicas = (
    df_rs_cleaned.groupby(df_rs_cleaned['cidade_limpa'].apply(lambda x: unidecode(x).lower()))['cidade_limpa']
    .first() # Mantém a versão mais bem formatada/acentuada
    .sort_values()
    .reset_index(drop=True)
)

total_cidades_higienizadas = len(df_cidades_unicas)

# 5. EXIBIR A LISTA HIGIENIZADA NO CONSOLE
print("=" * 65)
print(f"LISTA HIGIENIZADA DE CIDADES DO RS (TOTAL: {total_cidades_higienizadas})")
print("=" * 65)

for idx, cidade in enumerate(df_cidades_unicas, start=1):
    print(f"{idx:03d} - {cidade}")

print("=" * 65)

# 6. EXPORTAR O EXCEL OFICIAL TRATADO (COM A ABA "Cidades - RS")
df_export = pd.DataFrame({
    'Nº': range(1, total_cidades_higienizadas + 1),
    'Cidade': df_cidades_unicas,
    'Estado': 'RS'
})

nome_arquivo = 'Cidades_RS_Higienizado.xlsx'

with pd.ExcelWriter(nome_arquivo, engine='openpyxl') as writer:
    df_export.to_excel(writer, sheet_name='Cidades - RS', index=False)

print(f"\n✅ Planilha '{nome_arquivo}' gerada com a aba 'Cidades - RS' e cidades deduplicadas!")

# Download automático da planilha pronta
files.download(nome_arquivo)

LISTA HIGIENIZADA DE CIDADES DO RS (TOTAL: 211)
001 - Agudo
002 - Alecrim
003 - Alegrete
004 - Alvorada
005 - Antônio Prado
006 - Araricá
007 - Arroio Grande
008 - Arroio do Meio
009 - Arroio do Padre
010 - Arroio do Sal
011 - Bagé
012 - Balneário Pinhal
013 - Barra do Ribeiro
014 - Barros Cassal
015 - Barão de Cotegipe
016 - Bento Gonçalves
017 - Boa Vista do Buricá
018 - Boa Vista do Incra
019 - Bom Princípio
020 - Bom Progresso
021 - Bom Retiro do Sul
022 - Boqueirão do Leão
023 - Butiá
024 - Cacequi
025 - Cachoeira do Sul
026 - Cachoeirinha
027 - Camaquã
028 - Camargo
029 - Cambará do Sul
030 - Campo Bom
031 - Candelária
032 - Candiota
033 - Canela
034 - Canguçu
035 - Canoas
036 - Capela de Santana
037 - Capão da Canoa
038 - Carazinho
039 - Carlos Barbosa
040 - Casca
041 - Caxias do Sul
042 - Caçapava do Sul
043 - Cerrito
044 - Cerro Branco
045 - Cerro Largo
046 - Charqueadas
047 - Cidreira
048 - Ciríaco
049 - Colinas
050 - Constantina
051 - Coronel Bicaco
052 - Coxilha
053 - Criss

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
# ANEXO LEITURA DOS ARQUIVOS DE CLIENTES.CSV E CLIENTES.XLSX ALÉM DE AUTO.CSV
# Ver os nomes exatos de todas as colunas
print("Colunas do XLSX:", df_clientes_xlsx.columns.tolist())
print("Colunas do CSV:", df_clientes_csv.columns.tolist())
print("Colunas do AUTO CSV:", df_auto.columns.tolist())

Colunas do XLSX: ['id_client;age;monthly_income;city;state;gender;education_level']
Colunas do CSV: ['id_client', 'age', 'monthly_income', 'city', 'state', 'gender', 'education_level']
Colunas do AUTO CSV: ['id_client', 'auto_brand', 'auto_model', 'auto_year', 'auto_value', 'id_cliente']
